# Algoritmos de Classificação

## Importações

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import math
import statistics
import matplotlib.pyplot as plt
from collections import Counter
import random

## Algoritmos

In [2]:
class Algorithm:
    def __init__(self):
        self.confusion_matrix = None
        pass
    
    def fit(self):
        pass

    def predict(self):
        pass
    
    def create_confusion_matrix(self, previsoes, real):
        confusion_matrix = np.zeros((2, 2))

        for i, k in zip(previsoes, real):
            confusion_matrix[k][i] += 1

        return confusion_matrix
    
    def score(self, y_resultado, y_teste):
        self.confusion_matrix = self.create_confusion_matrix(y_resultado, y_teste)

        acuracia = np.sum(np.diagonal(self.confusion_matrix))/np.sum(self.confusion_matrix)
        precisao = (self.confusion_matrix[1][1]/(np.sum(self.confusion_matrix[:, 1])))
        revocacao = (self.confusion_matrix[1][1] / (np.sum(self.confusion_matrix[1][:])))
        f1_score = (2/((1/precisao) + 1/(revocacao)))

        # print("=" * 10 + " Medições de Desempenho " + "=" * 10)
        print("Matriz de Confusão")
        print(f"{pd.DataFrame(self.confusion_matrix, columns=['0', '1'])}")
        print(f"\nAcurácia: {acuracia:.4f}")
        print(f"Precisão: {precisao:.4f}")
        print(f"Revocação: {revocacao:.4f}")
        print(f"F1-Score: {f1_score:.4f}")
        # print("=" * 44)

In [3]:
class Preprocessor:
    def __init__(self, train, test):
        self.algorithm = None
        self.train = train
        self.test = test
        self.encoders = self.Encoders()

    def preprocessor(self):
        # self.algorithm = algorithm

        self.train = self.check_categorical_features(self.train)
        self.test = self.check_categorical_features(self.test)

        train_norm, test_norm = self.normalize(self.train[self.train.columns[:-1]].values, self.test[self.train.columns[:-1]].values)
        self.train[self.train.columns[:-1]] = train_norm
        self.test[self.train.columns[:-1]]  = test_norm
            
        return self.train, self.test

    def create_encoder(self, feature, nome):
        encoder = LabelEncoder()
        feature_codificada = encoder.fit_transform(feature)
        self.encoders.add_encoder(nome, encoder)
        return feature_codificada

    def check_categorical_features(self, data):
        for i in range(int(data.size/len(data)) - 1):
            feature = data.iloc[:, i]
            if not all(isinstance(instance, (int, float)) for instance in (feature.values).tolist()):
                y = self.encoders.get_encoder(feature.name)
                if y == None:
                    data.loc[:, feature.name] = self.create_encoder((feature.values).tolist(), feature.name)
                else:
                    data.loc[:, feature.name] = y.transform((feature.values).tolist())
        
        return data
    
    class Encoders:
        def __init__(self):
            self.encoders = {}

        def add_encoder(self, feature, encoder):
            self.encoders[feature] = encoder

        def get_encoder(self, feature):
            try:
                return self.encoders[feature]
            except KeyError:
                return None
    
    def normalize(self, x_train, x_test):
        normalizer = MinMaxScaler()
        normalizer.fit(x_train)
        return normalizer.transform(x_train), normalizer.transform(x_test)

### Naive Bayes

In [4]:
class NaiveBayes(Algorithm): 
    def __init__(self):
        self.bayesian_table = {}
        self.target_categories = None
        self.labels_features = None
        self.amount_target = None
        self.amount_features = None

    def __initialize_bayesian_table(self, x):
        features = [[] for _ in range(self.amount_features)]
        
        for i in range(len(x)):
            for j in range(self.amount_features):
                features[j].append(x[i][j])

        for i in range(len(features)):
            features[i] = np.unique(features[i])

        for i in self.target_categories:
            self.bayesian_table[i] = {}
        
            for j, k in enumerate(features):
                self.bayesian_table[i][j] = {}

                for l in k:
                    self.bayesian_table[i][j][l] = 0

    def __calculate_bayesian_table(self, x, y):
        for i, j in enumerate(x):
            for k in range(len(j)):
                self.bayesian_table[y[i]][k][j[k]] += 1
    
    def fit(self, x, y):
        counter_feature = np.unique(y, return_counts=True)
        self.amount_target = counter_feature[1]
        self.target_categories = counter_feature[0]
        self.amount_features = int(x.size/len(x))

        self.__initialize_bayesian_table(x)
        self.__calculate_bayesian_table(x, y)

        for i in self.target_categories:
            for k in self.bayesian_table[i]:
                for j in self.bayesian_table[i][k]:
                    self.bayesian_table[i][k][j] /= self.amount_target[i]

    def __calculate_probability(self, x):
        p = []
        for i in self.target_categories:
            product = 1
            for j, k in enumerate(x):
                product *= self.bayesian_table[i][j][k]
            
            p.append(product * self.amount_target[i]/np.sum(self.amount_target)) 

        for i, k in enumerate(p):
            if k == max(p):
                return self.target_categories[i]

    def predict(self, x):
        return list(map(self.__calculate_probability, x))

In [5]:
class Node:
    def __init__(self, type, feature = None, value = None):
        self.type = type
        self.feature = feature
        self.value = value
        self.connections = {}
        if feature != None: self.__initialize_connections()
    
    def __initialize_connections(self):
        for i in self.feature.attributes:
            self.connections[i] = None

In [6]:
class Component:
    def __init__(self, name, attributes, type, priori_prob = None, conditional_prob = None, entropy = None, gain = 0):
        self.name = name
        self.attributes = attributes
        self.type = type
        self.priori_prob = priori_prob
        self.conditional_prob = conditional_prob

        if type != 'target': 
            self.entropy = {}
        else: 
            self.entropy = entropy

        self.gain = gain
    
    def calculate_priori_prob(self, values_feature):
        buffer = dict(Counter(values_feature))
        sum_values = sum(buffer.values())

        for i in buffer:
            buffer[i] = buffer[i]/sum_values
            
        self.priori_prob = buffer
    
    def calculate_conditional_prob(self, target_attributes, target_values, values_feature):
        buffer = {i: {} for i in self.attributes}

        for i in buffer:
            buffer[i] = {j: 0 for j in target_attributes}

        for i, j in zip(values_feature, target_values):
            buffer[i][j] += 1

        for i in buffer:
            sum_values = sum(buffer[i].values())
            for j in buffer[i]:
                buffer[i][j] = buffer[i][j]/sum_values

        self.conditional_prob = buffer

In [7]:
class DecisionTree(Algorithm):
    def __init__(self, max_depth = float('inf')):
        self.root = None
        self.x = None
        self.y = None
        self.max_depth = max_depth
        self.depth = 0

    def __set_root(self, root):
        self.root = root

    def __calculate_entropy(self, s):
        entropy = 0
        for i in s:
            if i != 0:
                entropy -= i * math.log(i, 2)
        
        return entropy

    def __calculate_gain(self, feature, target):
        for i in feature.attributes:
            feature.gain += feature.entropy[i] * feature.priori_prob[i]
        
        feature.gain = target.entropy - feature.gain

    def __calculate_biggest_gain(self, features):
        earnings = [features[i].gain for i in features]
        
        for i in features:
            if features[i].gain == max(earnings):
                return features[i]
        
        return None

    def __calculate_new_database(self, node, connection, target, labels_features, current_base_indexes):
        indexes = []

        for i in current_base_indexes:
            if self.x.iloc[i][node.name] == connection:
                indexes.append(i)
        
        labels_x = [i for i in labels_features if i != node.name]
        metadata_x = [np.array(indexes), labels_x]
        metadata_y = [np.array(indexes), target]
        
        return metadata_x, metadata_y

    def __calculate_decision_tree(self, metadata_x, metadata_y):
        labels_features, label_target = metadata_x[1], metadata_y[1]
        
        x_train, y_train = self.x.iloc[metadata_x[0]][metadata_x[1]].values, np.array([self.y.values[i] for i in metadata_y[0]])

        if self.depth == self.max_depth:
            label = Counter(y_train).most_common(1)[0][0]
            return Node('sheet', None, label)

        target = Component(label_target, list(Counter(y_train).keys()), 'target')
        target.calculate_priori_prob(y_train)
        target.entropy = self.__calculate_entropy(list(target.priori_prob.values()))

        features = {}

        for i, j in enumerate(labels_features):
            features[j] = Component(j, list(Counter(x_train.T[i]).keys()), 'feature')
            features[j].calculate_conditional_prob(list(Counter(y_train).keys()), y_train, x_train.T[i])
            features[j].calculate_priori_prob(x_train.T[i])

            for i in features[j].attributes:
                features[j].entropy[i] = self.__calculate_entropy(list(features[j].conditional_prob[i].values()))

            self.__calculate_gain(features[j], target)

        feature = self.__calculate_biggest_gain(features)
        
        if feature.gain != 0 and self.depth < self.max_depth:
            self.depth += 1
            node = Node('node', feature)
            current_base_indexes = metadata_x[0]

            for i in node.connections:
                metadata_x, metadata_y = self.__calculate_new_database(node.feature, i, target, labels_features, current_base_indexes)
                node.connections[i] = self.__calculate_decision_tree(metadata_x, metadata_y)
            return node
        
        elif len(list(Counter(y_train).keys())) == 1:
            return Node('sheet', None, list(Counter(y_train).keys())[0])
        
        return Node('sheet')

    
    def __calculate_most_frequency_sheet(self, node):
        list = []
        for i in node.connections:
            j = node.connections[i]
            if(j.type == 'sheet' and j.value != None):
                list.append(j.value)

        return statistics.mode(list)
    
    def __check_none_sheets(self, node):
        list = []
        for i in node.connections:
            j = node.connections[i]
            if(j.type == 'sheet'):
                if(j.value == None):
                    j.value = self.__calculate_most_frequency_sheet(node)
            else:
                list.append(j)
        
        for i in list:
            self.__check_none_sheets(i)
        
    def __calculate_majority_class(self, node, list):
        for i in node.connections:
            j = node.connections[i]
            if(j.type == 'sheet'):
                list.append(j.value)
            else:
                self.__calculate_majority_class(j, list)
        return list
        
    def print_decision_tree(self, node):
        list = []

        print("==========================")
        print(f"Node: {node.feature.name}")

        for i in node.connections:
            j = node.connections[i]
            print(f"Attribute: {i} -> {j.type}")
            if(j.type != 'sheet'):
                print(f"\t -> {j.feature.name}")
                list.append(j)
            else:
                print(f"\tResult: {j.value}")

        print("==========================\n\n")

        for i in list:
            self.print_decision_tree(i)
    
    def fit(self, x_train, y_train):
        if not isinstance(x_train, pd.DataFrame) or not isinstance(y_train, pd.Series):
            print("Warning: the data must be a dataframe and a series, respectively!\n")
            return
        
        self.x, self.y = x_train, y_train

        metadata_x = [np.array(range(self.x.index.stop)), list(self.x.columns)]
        metadata_y = [np.array(range(self.y.index.stop)), self.y.name]

        self.__set_root(self.__calculate_decision_tree(metadata_x, metadata_y))
        self.__check_none_sheets(self.root)

    def __decision_tree(self, instance, node):
        result = None
        if node.type != 'sheet':
            for i in node.connections:
                if i == instance[node.feature.name]:
                    result = self.__decision_tree(instance, node.connections[i])
        else: 
            return node.value
        
        if result == None:
            list = []
            list = self.__calculate_majority_class(node, list)
            result = statistics.mode(list)

        return result

    def predict(self, x_test):
        results = []

        for i in x_test:
            results.append(self.__decision_tree(i, self.root))

        return results


In [8]:
class KNN(Algorithm):
    def __init__(self, k = 3):
        self.x = None
        self.y = None
        self.k = k

    def fit(self, x, y):
        self.x = x
        self.y = y

    def __calculate_nearest_neighbor(self, x):
        distances = []

        for index, i in enumerate(self.x):
            d = math.sqrt(sum([math.pow(l, 2) for l in (i - x)]))
            distances.append((d, index))

        distances = sorted(distances)
        shorter_distances = [distances[i] for i in range(self.k) ]
        classes = [self.y[j] for i, j in shorter_distances]
        return statistics.mode(classes)
        
    def predict(self, x):
        return list(map(self.__calculate_nearest_neighbor, x))

In [9]:
class LogisticRegression(Algorithm):
    def __init__(self, alpha = 1, l = 1, tolerancia = 0.000001):
        self.taxa_aprendizado = alpha
        self.param_regularizacao = l
        self.tolerancia = tolerancia
        self.qtd_features = None
        self.qtd_instancias = None
        self.parametros = None
        self.max_iteracoes = 1000
        self.qtd_iteracoes = 0
        self.log_loss_historico = []

    def fit(self, x, y):
        self.qtd_instancias = len(x)
        self.qtd_features = int(x.size/self.qtd_instancias)
        self.parametros = np.ones(self.qtd_features + 1).reshape(-1, 1)

        matriz_design = np.column_stack([np.ones(self.qtd_instancias), x])
        y = np.array(y).reshape(-1, 1)

        erro_relativo = 1000

        self.qtd_iteracoes = 0

        log_loss_anterior = 1000

        while(erro_relativo > self.tolerancia and self.qtd_iteracoes <= self.max_iteracoes):
            modelo_regressao_linear = (np.dot(matriz_design, self.parametros)).astype(np.float64)
            y_previsto = self.funcao_logistica(modelo_regressao_linear)
            y_previsto = (y_previsto).reshape(-1, 1)
            matriz_transposta = matriz_design.T

            vetor_gradiente = np.dot(matriz_transposta, y_previsto - y)/self.qtd_instancias

            penalidade_ridge = (self.parametros.copy() * self.param_regularizacao)/self.qtd_instancias
            penalidade_ridge[0] = 0

            self.parametros = self.parametros - (self.taxa_aprendizado) * (vetor_gradiente + penalidade_ridge)

            log_loss_atual = self.log_loss(x, y)

            erro_relativo = abs(log_loss_anterior - log_loss_atual)/log_loss_anterior

            log_loss_anterior = log_loss_atual
            self.log_loss_historico.append(log_loss_atual)

            self.qtd_iteracoes += 1

    def log_loss(self, x, y_real):
        matriz_design = np.column_stack([np.ones(len(x)), x])
        regressao_linear = (np.dot(matriz_design, self.parametros)).astype(np.float64)
        y_previsto = self.funcao_logistica(regressao_linear)

        valor_minimo = 1e-15
        y_previsto = np.clip(y_previsto, valor_minimo, 1 - valor_minimo)

        loss = -np.mean(y_real * np.log(y_previsto) + (1 - y_real) * np.log(1 - y_previsto))
        penalidade_ridge = (self.param_regularizacao/2)*np.sum(self.parametros[1:] ** 2)/len(x)
        loss = loss + penalidade_ridge

        return loss

    def funcao_logistica(self, t):
        return 1/(1 + np.exp(-t))
    
    def prob_logistica(self, p, threshold):
        return 1 if p >= threshold else 0
    
    def predict(self, x, threshold = 0.5):
        self.qtd_instancias = len(x)

        matriz_design = np.column_stack([np.ones(self.qtd_instancias), x])
        regressao_linear = (np.dot(matriz_design, self.parametros)).astype(np.float64)
        y_previsto = self.funcao_logistica(regressao_linear)

        return list(map(lambda y: self.prob_logistica(y, threshold), y_previsto))

    def curva_log_loss(self):
        iteracoes = np.arange(self.qtd_iteracoes)
        plt.plot(iteracoes, self.log_loss_historico)
        plt.xlabel("Iterações")
        plt.ylabel("Custo (log loss)")
        plt.title("Curva de Convergência")
        plt.show()

    
    def roc_curve(self, x_teste, y_teste):
        threshold = np.arange(start=0, stop=1, step=0.01)
        predicoes = list((map(lambda t:  self.predict(x_teste, t), threshold)))
        
        taxa_verdadeiros_positivos, taxa_falsos_positivos = [], []

        for i in range(len(predicoes)):
            m = self.confusion_matrix(predicoes[i], y_teste)
            tpr = (m[1][1]/(np.sum(m[1][:])))

            if np.sum(m[0][:]) != 0: fpr = (m[0][1]/(np.sum(m[0][:])))
            else: fpr = 0

            taxa_verdadeiros_positivos.append(tpr)
            taxa_falsos_positivos.append(fpr)

        plt.plot(taxa_falsos_positivos, taxa_verdadeiros_positivos)
        plt.xlabel("Taxa de Falsos Positivos")
        plt.ylabel("Taxa de Verdadeiros Positivos")
        plt.title("Curva ROC")
        plt.show()


In [10]:
class SVMLinear(Algorithm):
    def __init__(self, alpha = 1, b = 0, c = 0.2, tolerancia = 0.000001):
        self.taxa_aprendizado = alpha
        self.param_b = b
        self.param_c = c
        self.tolerancia = tolerancia
        self.max_iteracoes = 1000
        self.qtd_features = None
        self.qtd_instancias = None
        self.parametros = None
        self.qtd_iteracoes = 0
        self.historico_funcao_custo = []

    def fit(self, x, y):
        self.qtd_instancias = len(x)
        self.qtd_features = int(x.size/self.qtd_instancias)
        self.parametros = np.ones(self.qtd_features).reshape(-1, 1)

        y = np.where(y == 0, -1, 1).reshape(-1, 1)

        erro_relativo = 1000

        self.qtd_iteracoes = 0
        custo_anterior = 1000

        while(erro_relativo > self.tolerancia and self.qtd_iteracoes <= self.max_iteracoes):
            modelo_svm = np.dot(x, self.parametros) + self.param_b

            classificacao_confianca = y * modelo_svm
            mascara_gradiente = np.where(classificacao_confianca < 1, 1, 0)

            vetor_gradiente_parametros = (self.parametros + self.param_c * np.dot(x.T, (-y * mascara_gradiente)))/self.qtd_instancias
            vetor_gradiente_b = self.param_c * np.sum(-y * mascara_gradiente) / self.qtd_instancias

            self.parametros = self.parametros - (self.taxa_aprendizado * vetor_gradiente_parametros)
            self.param_b -= (vetor_gradiente_b) * (self.taxa_aprendizado)

            custo_atual = self.funcao_custo_svm_linear(classificacao_confianca)

            erro_relativo = abs(custo_anterior - custo_atual)/custo_anterior

            custo_anterior = custo_atual
            self.historico_funcao_custo.append(custo_atual)

            self.qtd_iteracoes += 1

    def funcao_custo_svm_linear(self, classificacao_confianca):
        return ((1/2)*(np.sum(self.parametros ** 2)) + self.param_c * np.sum(np.maximum(0, 1 - classificacao_confianca)))/self.qtd_instancias

    def predict(self, x):
        modelo_svm = (np.dot(x, self.parametros)) + self.param_b
        return np.where(modelo_svm < 0, 0, 1).flatten()

    def curva_convergencia(self):
        iteracoes = np.arange(self.qtd_iteracoes)
        plt.plot(iteracoes, self.historico_funcao_custo)
        plt.xlabel("Iterações")
        plt.ylabel("Custo SVM Linear")
        plt.title("Curva de Convergência")
        plt.show()

In [11]:
class SimplePerceptron(Algorithm):
    def __init__(self, alpha = 1, max_epocas = 10000):
        self.pesos = None
        self.entradas = None
        self.saidas = None
        self.taxa_aprendizado = alpha
        self.max_epocas = max_epocas
        self.qtd_epocas = 0
        self.historico_erros = []

    def __aleatoriza_pesos(self, r, qtd_pesos):
        self.pesos = np.empty(qtd_pesos + 1)
        
        for i in range(qtd_pesos + 1):
            self.pesos[i] = (random.triangular(-r, r))

    def __aleatoriza_dataset(self):
        indices = np.arange(len(self.entradas))
        np.random.shuffle(indices)
        
        self.entradas = self.entradas[indices]
        self.saidas = self.saidas[indices]

    def __calcula_saida_atual(self, x):
        u = np.dot(x, self.pesos)

        if u >= 0: 
            return 1
        
        return 0

    def __perceptron(self):
        erro_total = 1000
        self.__aleatoriza_pesos(1, int(self.entradas.size/len(self.entradas)))

        while(erro_total > (len(self.entradas) * 0.07) and self.max_epocas >= self.qtd_epocas):
            self.__aleatoriza_dataset()
            matriz_design = np.column_stack([np.ones(len(self.entradas)) * -1, self.entradas])
            self.qtd_epocas += 1
            erro_total = 0

            for i, x in enumerate(matriz_design):
                y = self.__calcula_saida_atual(x)
                erro = self.saidas[i] - y
                pesos_atuais = self.pesos + self.taxa_aprendizado * erro * x
                erro_total += int(erro != 0)
                self.pesos = pesos_atuais
            
            self.historico_erros.append(erro_total)

    def curva_convergencia(self):
        iteracoes = np.arange(self.qtd_epocas)
        plt.plot(iteracoes, self.historico_erros)
        plt.xlabel("Iterações")
        plt.ylabel("Erro")
        plt.title("Curva de Convergência")
        plt.show()
    
    def fit(self, x, y):
        self.entradas = x
        self.saidas = y
        self.__perceptron()

    def predict(self, x):
        matriz_design = np.column_stack([np.ones(len(x)) * -1, x])
        return list(map(self.__calcula_saida_atual, matriz_design))

In [12]:
train = pd.read_csv('dataset/diabetes_train.csv')
test = pd.read_csv('dataset/diabetes_test.csv')

In [13]:
x_train, x_test = train.iloc[:, :-1], test.iloc[:, :-1]

In [14]:
y_train, y_test = train.iloc[:, -1], test.iloc[:, -1]

In [15]:
naive_bayes = NaiveBayes()
decision_tree = DecisionTree()
knn = KNN()
logistic_regression = LogisticRegression()
svm = SVMLinear()
simple_perceptron = SimplePerceptron()

In [16]:
naive_bayes.fit(x_train.values, y_train.values)
decision_tree.fit(x_train, y_train)
resultados_naive_bayes = naive_bayes.predict(x_test.values)
results_decision_tree = decision_tree.predict(x_test.to_dict('records'))

In [17]:
preprocessor = Preprocessor(train, test)
train, test = preprocessor.preprocessor()
x_train, x_test = train.iloc[:, :-1].values, test.iloc[:, :-1].values
y_train, y_test = train.iloc[:, -1].values, test.iloc[:, -1].values

In [26]:
knn.fit(x_train, y_train)
results_knn = knn.predict(x_test)
logistic_regression.fit(x_train, y_train)
results_logistic_regression = logistic_regression.predict(x_test)
svm.fit(x_train, y_train)
results_svm = svm.predict(x_test)
simple_perceptron.fit(x_train, y_train)
results_simple_preceptron = simple_perceptron.predict(x_test)

In [27]:
decision_tree.score(results_decision_tree, y_test)

Matriz de Confusão
      0      1
0  79.0    2.0
1  10.0  117.0

Acurácia: 0.9423
Precisão: 0.9832
Revocação: 0.9213
F1-Score: 0.9512


In [20]:
naive_bayes.score(resultados_naive_bayes, y_test)

Matriz de Confusão
      0      1
0  74.0    7.0
1  17.0  110.0

Acurácia: 0.8846
Precisão: 0.9402
Revocação: 0.8661
F1-Score: 0.9016


In [21]:
knn.score(results_knn, y_test)

Matriz de Confusão
      0      1
0  79.0    2.0
1   5.0  122.0

Acurácia: 0.9663
Precisão: 0.9839
Revocação: 0.9606
F1-Score: 0.9721


In [22]:
logistic_regression.score(results_logistic_regression, y_test)

Matriz de Confusão
      0      1
0  74.0    7.0
1   9.0  118.0

Acurácia: 0.9231
Precisão: 0.9440
Revocação: 0.9291
F1-Score: 0.9365


In [23]:
svm.score(results_svm, y_test)

Matriz de Confusão
      0      1
0  73.0    8.0
1   8.0  119.0

Acurácia: 0.9231
Precisão: 0.9370
Revocação: 0.9370
F1-Score: 0.9370


In [28]:
simple_perceptron.score(results_simple_preceptron, y_test)

Matriz de Confusão
      0      1
0  73.0    8.0
1   9.0  118.0

Acurácia: 0.9183
Precisão: 0.9365
Revocação: 0.9291
F1-Score: 0.9328
